# SSM2603 codec configuration demo

Bring up the Zybo's **SSM2603 audio codec** from Python using the
`sampler_codec_controller` IP in the PL. The controller is a custom AXI4-Lite
core that does the I2C to the codec; we only poke its registers and poll status
bits. The driver (`CodecController`) is a Python port of the original FreeRTOS
baremetal code in `subsystems/codec_unit/fw/` — see `codec_controller.py` there.

> **Run this as root.** This board uses the custom non-XRT PYNQ backend
> (`zynq_cma_device`) that does MMIO over `/dev/mem`, which needs root. The
> bitstream is already loaded at boot (it lives in `boot.bin`), so the PL is live
> and we only *attach* to it — we never reprogram it.

**Controller register map** (from `codec_controller_control_regs.h`, byte offsets):

| offset | register | notes |
|--------|----------|-------|
| `0x00` | `CTRL`    | bit0 wr, bit1 rd, bit2 busy(RO), bit3 init_done, bit4 data_valid, bit5 missed_ack, bit31 reset |
| `0x04` | `ADDR`    | SSM2603 register address |
| `0x08` | `WR_DATA` | data to write |
| `0x0C` | `RD_DATA` | data read back (HW written; resets to `0xcafecafe`) |

Every `CTRL` bit ignores written 0s, so the driver never read-modify-writes it —
it writes a single action bit at a time (see `codec_registers.sv`).

## 1. Imports and driver

`zynq_cma_device` must be imported **before** `pynq` — importing it registers a
`/dev/mem`-based device as the active PYNQ device so `Overlay`/`MMIO` work without
XRT. We add the firmware directory to `sys.path` so we can import the ported
`CodecController`.

In [ ]:
import zynq_cma_device            # registers itself as the active PYNQ device
from pynq import Overlay

import os
import sys

# Make the ported driver importable. It lives next to the original C source in
# subsystems/codec_unit/fw/. Try a few likely locations relative to the notebook.
for _cand in (
    os.path.join(os.getcwd(), '..', 'subsystems', 'codec_unit', 'fw'),
    os.path.join(os.getcwd(), 'codec_unit', 'fw'),
    os.getcwd(),  # if codec_controller.py was staged next to this notebook
):
    _cand = os.path.abspath(_cand)
    if os.path.exists(os.path.join(_cand, 'codec_controller.py')):
        sys.path.insert(0, _cand)
        break

from codec_controller import CodecController, CTRL, RD_DATA

# The codec sits in the same design as the FFT demo; the bit/hwh pair is staged
# next to this notebook by the gen-sdcard-image make target.
BITSTREAM = 'fft_demo_top_wrapper.bit'

## 2. Attach to the overlay

The PL is already programmed, so we pass `download=False` — PYNQ just parses the
`.hwh` to build the IP driver map without touching the running design. The codec
controller is exposed as `sampler_codec_controller` (it's the `codec_controller`
cell inside the `sampler` hierarchy of the block design).

In [ ]:
ol = Overlay(BITSTREAM, download=False)   # attach only; don't reprogram the PL
print(sorted(ol.ip_dict.keys()))

codec = CodecController(ol.sampler.codec_controller)
codec

## 3. Sanity check the controller

Before touching the codec, confirm we can reach the controller registers. The
`RD_DATA` register resets to the sentinel `0xcafecafe`, and the `CTRL` register
should read back with the busy bit clear (bit 2 = 0).

In [ ]:
ctrl = codec.ip.read(CTRL)
print(f'CTRL    = 0x{ctrl:08x}  (busy={bool(ctrl & (1 << 2))})')
print(f'RD_DATA = 0x{codec.ip.read(RD_DATA):08x}  (0xcafecafe = never read yet)')

## 4. Configure the codec

`init()` runs the full bring-up sequence (port of `vCodecInit`): software reset,
power up chip/DAC/ADC/line-in, DSP master mode, ~44.1 kHz (USB mode, 256×fs),
route DAC to output, unmute, set volumes, activate the digital core, then enable
the output. Each step reads the register back (`check=True`) and raises on a
mismatch, so if this cell completes without error the codec is configured.

In [ ]:
codec.init()

## 5. Verify by reading registers back

Read a few key SSM2603 registers through the controller and confirm they hold the
values `init()` wrote. Reads go through the same I2C controller (`codec_rd`).

In [ ]:
from codec_controller import (
    R_POWER, R_IFACE, R_SAMPLE, R_ANALOG, R_ACTIVE, R_LHPVOL,
)

checks = {
    'POWER  (0x06)': (R_POWER,  0x22),
    'IFACE  (0x07)': (R_IFACE,  0x53),
    'SAMPLE (0x08)': (R_SAMPLE, 0x23),
    'ANALOG (0x04)': (R_ANALOG, 0x10),
    'ACTIVE (0x09)': (R_ACTIVE, 0x01),
}
for name, (addr, expected) in checks.items():
    got = codec.codec_rd(addr)
    flag = 'ok' if got == expected else 'MISMATCH'
    print(f'{name}: read 0x{got:02x}  expected 0x{expected:02x}  [{flag}]')

## 6. Adjust the volume

`set_output_volume` takes dB (headphone/DAC, `0x79` == 0 dB internally, 1 dB per
step) and updates both channels at once. `set_input_volume` takes the raw 6-bit
ADC code (`0x17` == 0 dB).

In [ ]:
codec.set_output_volume(-6)          # a bit louder than the -15 dB init default
print(f'LHPVOL (0x02) = 0x{codec.codec_rd(R_LHPVOL):02x}')

codec.set_input_volume(0x17)         # 0 dB ADC input

## 7. Manual register access

For poking individual registers (the equivalent of the old `codec_reg <addr>
[data]` CLI command). Writes can `check=True` to verify via read-back.

In [ ]:
# Read one register
print(f'power mgmt = 0x{codec.codec_rd(R_POWER):02x}')

# Write one register with read-back verification
codec.codec_wr(R_POWER, 0x22, check=True)
print('write + read-back ok')